In [32]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

df = pd.read_csv("premier_league_stats.csv")
missing_age_df = df[df['age'].isna()]
print(f"Found {len(missing_age_df)} players with missing ages")

driver = webdriver.Firefox()
wait = WebDriverWait(driver, 5)
cookie_accepted = False

try:
    for idx, row in missing_age_df.iterrows():
        player_name = row['player']
        print(f"\nSearching for: {player_name}")
        
        driver.get("https://www.transfermarkt.com")
        time.sleep(3)
        
        if not cookie_accepted:
            try:
                accept_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept') or contains(text(), 'Agree')]")))
                accept_button.click()
                cookie_accepted = True
                time.sleep(2)
            except:
                pass
        
        search_box = wait.until(EC.presence_of_element_located((By.NAME, "query")))
        search_box.clear()
        search_box.send_keys(player_name)
        
        search_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "tm-header__input--search-send")))
        # use click to avoid cookies
        driver.execute_script("arguments[0].click();", search_button)
        time.sleep(3)
        
        try:
            age_element = wait.until(EC.presence_of_element_located((By.XPATH, "//table[@class='items']//tbody//tr[1]//td[@class='zentriert'][3]")))
            age = age_element.text.strip()
            print(f"Found age: {age}")
            df.at[idx, 'age'] = int(age)  
        except Exception as e:
            print(f"Could not find age for {player_name}: {e}")
        
        time.sleep(3)
    
    df.to_csv("premier_league_stats_updated.csv", index=False)
    print("\nUpdated CSV saved!")

finally:
    driver.quit()

print("\nDone!")

Found 5 players with missing ages

Searching for: Mateus Mane
Found age: 18

Searching for: Jake Evans
Found age: 17

Searching for: Olabade Aluko
Found age: 18

Searching for: Tom Taylor
Found age: 20

Searching for: Jayden Moore
Found age: 18

Updated CSV saved!

Done!


In [33]:

df.isnull().sum()

player             0
nationality        6
position           0
age                0
games              0
games_starts       0
minutes          128
minutes_90s      128
goals            128
assists          128
goals_assists    128
goals_pens       128
pens_made        128
pens_att         128
cards_yellow     128
cards_red        128
team               0
dtype: int64

In [34]:
player_nationality_list = [
    ("Mateus Mane", "eng ENG"),
    ("Jeremy Monga", "eng ENG"),
    ("Jake Evans", "can CAN"), 
    ("Olabade Aluko", "eng ENG"),
    ("Tom Taylor", "eng ENG"),
    ("Jayden Moore", "eng ENG")
]


for key , value in player_nationality_list:
    df.loc[df['player'] == key, 'nationality'] = value

#filling the null values of the stats with 0
df.fillna(0 , inplace=True)


In [35]:
df.isna().sum()

df.to_csv("premier_league_stats_updated.csv", index=False)
print("\nUpdated CSV saved!")


Updated CSV saved!
